## NVIDIA DLI at ModelScope Notebook

原课程：NVIDIA DLI《深度学习基础》，来源 [https://github.com/NVDLI/fundamentals-of-deep-learning-zh](https://github.com/NVDLI/fundamentals-of-deep-learning-zh) 。本版本由 VoyagerX 适配，原课程正文和练习保留。模型与数据集从 VoyagerX 的 ModelScope 仓库下载，不访问 Hugging Face、Kaggle 或 PyTorch 外部下载站。


In [ ]:
import os, sys
from pathlib import Path
tutorial_dir = Path("/mnt/workspace/fundamentals-of-deep-learning-zh/tutorials")
if not tutorial_dir.is_dir():
    # 首次运行下载课程目录；跳过空链接并打印对应文件名。
    !python -c 'import requests; from pathlib import Path; from concurrent.futures import ThreadPoolExecutor; b="https://modelscope.cn"; gid="cb1503c2-1458-4f67-b9c3-bd505091cb47"; dst=Path("/mnt/workspace/fundamentals-of-deep-learning-zh"); r=requests.get(b+"/api/v1/gallery",params={"Gid":gid},timeout=60); r.raise_for_status(); files=r.json()["Data"]["Gallery"]["Files"]; r=requests.post(b+"/api/v1/gallery/square/files/download",json={"Gid":gid,"FileNames":[f for f in files if f.startswith("course_content/")],"Draft":False},timeout=60); r.raise_for_status(); urls=r.json()["Data"]["Urls"]; exec("def download(x):\n u=x.get(\"Url\")\n if not u:\n  print(\"跳过空下载链接：\",x.get(\"Filename\")); return\n p=dst/x[\"Filename\"].removeprefix(\"course_content/\")\n p.parent.mkdir(parents=True,exist_ok=True)\n r=requests.get(u,timeout=300)\n r.raise_for_status()\n p.write_bytes(r.content)"); pool=ThreadPoolExecutor(max_workers=6); list(pool.map(download,urls)); pool.shutdown(); print("有效链接下载完成：",dst)'
os.chdir(tutorial_dir); sys.path.insert(0, str(tutorial_dir))
from modelscope_assets import prepare_bert


# 6. 自然语言处理

在这个教程中，我们将离开像静态图像这样的独立数据，转向依赖于序列中其它数据项的数据。作为示例，我们将使用句子文本。语言自然由序列数据组成，形式为单词中的字符和句子中的单词。其它序列数据的例子包括股票价格和天气数据等时间序列。视频虽然包含静态图像，但也属于序列。当数据中的元素与之前和之后的内容有关系，就需要采取不同的方法来处理。

## 6.1 目标

* 使用分词器为神经网络准备文本
* 观察如何使用嵌入来识别文本数据的数值特征

## 6.2 BERT 

BERT，全称为 **B**idirectional **E**ncoder **R**epresentations from **T**ransformers，是 [Google](https://www.google.com/) 在 2018 年引入的一种开创性模型。

BERT 同时针对两个目标进行训练：
* 从一系列单词中预测缺失的单词
* 在一系列句子之后预测新的句子

让我们用这两种挑战来看看 BERT 的表现。

## 6.3 分词

由于神经网络是数值计算机，让我们将文本转换为数值 token。让我们加载 BERT 的[分词器](https://huggingface.co/docs/transformers/main_classes/tokenizer#tokenizer)：

In [ ]:
import torch
from transformers import BertTokenizer, BertForMaskedLM, BertForQuestionAnswering
tokenizer = BertTokenizer.from_pretrained(prepare_bert("bert-base-cased"), local_files_only=True)

BERT 的`分词器`可以一次性[编码](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.PreTrainedTokenizer.encode)多段文本。稍后我们将测试 BERT 的记忆力，先给它一些信息和一个关于信息的问题。您可以随时回到这里，尝试不同的句子组合。

In [ ]:
text_1 = "I understand equations, both the simple and quadratical."
text_2 = "What kind of equations do I understand?"

# Tokenized input with special tokens around it (for BERT: [CLS] at the beginning and [SEP] at the end)
indexed_tokens = tokenizer.encode(text_1, text_2, add_special_tokens=True)
indexed_tokens

如果计算 token 的数量，会发现句子中的 token 数量多于单词数。让我们来看看为什么会这样。可以用 [convert_ids_to_tokens](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.PreTrainedTokenizer.convert_ids_to_tokens) 来查看使用了哪些 token。

In [ ]:
tokenizer.convert_ids_to_tokens([str(token) for token in indexed_tokens])

索引列表比原始输入长的原因有两个：
1. `tokenizer` 添加了 `special_tokens` 来表示序列的开始（`[CLS]`）和句子之间的分隔（`[SEP]`）。
2. `tokenizer` 可以将一个词分解成多个部分。

从语言学角度来看，第二点很有趣。许多语言都有[词根](https://zh.wikipedia.org/wiki/%E8%AF%8D%E6%A0%B9)，或构成单词的组成部分。例如，“quadratic”这个词的词根是“quadr”，意思是“4”。BERT 并不是用语言定义词根，而是使用 [WordPiece](https://paperswithcode.com/method/wordpiece) 模型来找出如何拆分单词模式。我们今天使用的 BERT 模型有一个 `28996` 个 token 的词汇表。

我们可以直接[解码](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.PreTrainedTokenizer.decode)编码过的文本。注意 `special_tokens` 已经被添加进去了。

In [ ]:
tokenizer.decode(indexed_tokens)

## 6.4 文本分段

为了使用 BERT 模型进行预测，它还需要一个 `segment_ids` 的列表。这是一个与 token 相同长度的向量，表示每个句子属于哪个段落。

由于我们的 `tokenizer` 添加了一些 `special_tokens`，可以使用这些特殊标记来找到段落。首先，定义哪个索引对应哪个特殊标记。

In [ ]:
cls_token = 101
sep_token = 102

接下来创建一个 `for` 循环。将从 `segment_id` 设置为 `0` 开始，并且每当看到 [SEP] 标记时就增加 `segment_id`。为了确保效果，稍后将把这些 `segment_ids` 和 `indexed_tokens` 作为张量输入模型。

In [ ]:
def get_segment_ids(indexed_tokens):
    segment_ids = []
    segment_id = 0
    for token in indexed_tokens:
        if token == sep_token:
            segment_id += 1
        segment_ids.append(segment_id)
    segment_ids[-1] -= 1  # Last [SEP] is ignored
    return torch.tensor([segment_ids]), torch.tensor([indexed_tokens])

测试一下。每个数字是否正确地对应第一句和第二句？

In [ ]:
segments_tensors, tokens_tensor = get_segment_ids(indexed_tokens)
segments_tensors

## 6.4 文本掩码（Text Masking）

先看看 BERT 对单词的处理。为了训练词嵌入，BERT 在一系列单词中掩掉一个单词。掩码用了一个特殊的标记：

In [ ]:
tokenizer.mask_token

In [ ]:
tokenizer.mask_token_id

让我们从之前的两个句子中选择位置索引 `5` 进行掩码。随时回到这里改变索引，看看结果会如何变化！

In [ ]:
masked_index = 5

接下来，应用掩码并验证它是否出现在句子序列中。

In [ ]:
indexed_tokens[masked_index] = tokenizer.mask_token_id
tokens_tensor = torch.tensor([indexed_tokens])
tokenizer.decode(indexed_tokens)

然后，加载用于预测缺失单词的模型：`modelForMaskedLM`。

In [ ]:
masked_lm_model = BertForMaskedLM.from_pretrained(prepare_bert("bert-base-cased"), local_files_only=True)

就像使用其它 PyTorch 模块一样，我们可以检查其架构。

In [ ]:
masked_lm_model

您能找到标有 `word_embeddings` 的部分吗？这些是 BERT 为每个 token 学习到的嵌入。

In [ ]:
embedding_table = next(masked_lm_model.bert.embeddings.word_embeddings.parameters())
embedding_table

我们可以验证 BERT 词汇表中的 `28996` 个 token 都有一个大小为 `768` 的嵌入。

In [ ]:
embedding_table.shape

测试一下模型吧！它能正确预测我们提供的句子中缺失的单词吗？我们将使用 [torch.no_grad](https://pytorch.org/docs/stable/generated/torch.no_grad.html) 来让 PyTorch 不计算梯度。

In [ ]:
with torch.no_grad():
    predictions = masked_lm_model(tokens_tensor, token_type_ids=segments_tensors)
predictions

这有点难读懂，来看一下 `shape`。

In [ ]:
predictions[0].shape

`24` 指的是 token 数，`28996` 是指对 BERT 词汇表中每个 token 的预测。我们想找到词汇表中所有 token 的最大值，可以用 [torch.argmax](https://pytorch.org/docs/stable/generated/torch.argmax.html)。

In [ ]:
# Get the predicted token
predicted_index = torch.argmax(predictions[0][0], dim=1)[masked_index].item()
predicted_index

让我们看看 token `1241` 对应的是什么：

In [ ]:
predicted_token = tokenizer.convert_ids_to_tokens([predicted_index])[0]
predicted_token

您觉得怎么样？正确吗？

In [ ]:
tokenizer.decode(indexed_tokens)

## 6.5 问题与回答

虽然词掩码很有趣，但 BERT 是为更复杂的问题设计的，比如句子预测。它能通过 [Attention Transformer](https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf) 架构来完成这一任务。

这部分将使用 BERT 的不同版本，它有自己的 tokenizer。让我们为示例句子找到一组新的 token。

In [ ]:
text_1 = "I understand equations, both the simple and quadratical."
text_2 = "What kind of equations do I understand?"

question_answering_tokenizer = BertTokenizer.from_pretrained(prepare_bert("bert-large-uncased-whole-word-masking-finetuned-squad"), local_files_only=True)
indexed_tokens = question_answering_tokenizer.encode(text_1, text_2, add_special_tokens=True)
segments_tensors, tokens_tensor = get_segment_ids(indexed_tokens)

接下来加载 `question_answering_model`。

In [ ]:
question_answering_model = BertForQuestionAnswering.from_pretrained(prepare_bert("bert-large-uncased-whole-word-masking-finetuned-squad"), local_files_only=True)

可以像在掩掉单词时一样输入 tokens 和 segments。

In [ ]:
# Predict the start and end positions logits
with torch.no_grad():
    out = question_answering_model(tokens_tensor, token_type_ids=segments_tensors)
out

`question_answering_model` 和问答模型正在扫描我们的输入序列，以找到最能回答问题的子序列。数值越高，答案就越有可能是从这里开始的。

In [ ]:
out.start_logits

同样，`end_logits` 中的数越高，答案就越可能结束在那个 token 上。

In [ ]:
out.end_logits

然后可以用 [torch.argmax](https://pytorch.org/docs/stable/generated/torch.argmax.html) 来找到从开始到结束的 `answer_sequence`：

In [ ]:
answer_sequence = indexed_tokens[torch.argmax(out.start_logits):torch.argmax(out.end_logits)+1]
answer_sequence

最后，[解码](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.PreTrainedTokenizer.decode)这些 token，看看答案是否正确！

In [ ]:
question_answering_tokenizer.convert_ids_to_tokens(answer_sequence)

In [ ]:
question_answering_tokenizer.decode(answer_sequence)

## 6.7 总结

干得好！您成功地使用了大语言模型 (LLM) 从一系列句子中提取答案。尽管 BERT 在首次发布时是最先进的，但许多其它 LLM 自那以后也取得了突破。[build.nvidia.com](https://build.nvidia.com/explore/discover) 上托管了许多这样的模型，可以在浏览器中与它们互动。去探索一下，了解了解当今的最先进技术吧！

### 6.7.1 清理内存
继续进行后续内容之前，请执行以下单元清理 GPU 显存。

In [ ]:
import IPython
app = IPython.Application.instance()
app.kernel.do_shutdown(True)

### 6.7.2 下一步

恭喜您，您已经完成了所有课程学习目标！

作为最后的练习，请在评估中成功完成一个端到端的图像分类问题，完成后您将获得本课程的证书。